In [ ]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img
from pathlib import Path
from atlasreader import create_output

In [ ]:
virtueORvice = 'virtues' # vices, virtues

stories = ["tunnel","21styear","prettymouth","shapessocial","shapesphysical"]

# output directory
base_dir1 = os.path.expanduser(
    "~/Library/CloudStorage/GoogleDrive-s.h.p.collin@tilburguniversity.edu/My Drive/research/ongoing/2024_shapesNarrativeMRI/__narratives_and_values_fMRI/wholeBrain_results/result_interpretation/"
)

output_dir = Path(f"{base_dir1}{virtueORvice}/minStatConjunction_ofMinimumStatConjunctionImages/")


In [ ]:
# Collect z-maps
z_maps = []
for story in stories:
    z_map_path = os.path.join(output_dir, story, "zscores_fcontrast_secondlevel.nii.gz")
    if os.path.exists(z_map_path):
        print(f"✅ Loaded {z_map_path}")
        z_maps.append(image.load_img(z_map_path))
    else:
        print(f"⚠️ Missing: {z_map_path}")

if len(z_maps) < 2:
    raise ValueError("Need at least two z-maps to compute a minimum-statistic conjunction.")

print(z_maps)

# Resample all to the first image’s space
target_img = z_maps[0]
z_maps_resampled = [image.resample_to_img(img, target_img, interpolation="continuous") for img in z_maps]

# Create dictionary dynamically
img_dict = {f"img{i+1}": z_maps_resampled[i] for i in range(len(z_maps_resampled))}
expr = "np.minimum.reduce((" + ", ".join(img_dict.keys()) + "))"

# Compute conjunction map
min_stat_map = image.math_img(expr, **img_dict)

# Save output
output_path = os.path.join(output_dir, "minimum_stat_conjunction.nii.gz")
min_stat_map.to_filename(output_path)
print(f"✅ Saved conjunction map to: {output_path}")

In [ ]:

thresholded_map, threshold = threshold_stats_img(
    stat_img=min_stat_map,
    alpha=0.05,            # FDR level (start here)
    height_control='fdr',  # use 'bonferroni' for more conservative FWER control
    cluster_threshold=0,   # use >0 to filter small clusters
    two_sided=True        # or True if you want two-sided test
)

print(threshold)

output_path = os.path.join(output_dir, "min_stat_conjunction_fdrcorrected.nii.gz")
thresholded_map.to_filename(output_path)

In [ ]:


# Display the thresholded statistical map on a glass brain
display = plotting.plot_glass_brain(
    thresholded_map,
    threshold=None,         
    colorbar=True,
    display_mode='lyrz',     # Combine lateral, coronal, sagittal, axial views
    plot_abs=False,          # Set True if you want both positive & negative activations
    cmap='cold_hot',         # Common for statistical maps
    title='FDR-corrected',
)

# Save the glass brain figure as an image
fig_path = os.path.join(output_dir, "glassbrain_min_stat_conjunction_fdrcorrected.png")
display.savefig(fig_path, dpi=300)
display.close()

print(f"Glass brain saved at: {fig_path}")

In [ ]:

thresholded_map, threshold = threshold_stats_img(
    stat_img=min_stat_map,
    alpha=0.001,            # FDR level (start here)
    height_control='fpr',  # use 'bonferroni' for more conservative FWER control
    cluster_threshold=10,   # use >0 to filter small clusters
    two_sided=True        # or True if you want two-sided test
)

output_path = os.path.join(output_dir, "min_stat_conjunction_fprcorrected_clusterThr10.nii.gz")
thresholded_map.to_filename(output_path)


In [ ]:


# Display the thresholded statistical map on a glass brain
display = plotting.plot_glass_brain(
    thresholded_map,
    threshold=None,         
    colorbar=True,
    display_mode='lyrz',     # Combine lateral, coronal, sagittal, axial views
    plot_abs=False,          # Set True if you want both positive & negative activations
    cmap='cold_hot',         # Common for statistical maps
    title='FPR-corrected (α=0.001, cluster>10)',
)

# Save the glass brain figure as an image
fig_path = os.path.join(output_dir, "glassbrain_min_stat_conjunction_fprClusterThr10.png")
display.savefig(fig_path, dpi=300)
display.close()

print(f"Glass brain saved at: {fig_path}")


In [ ]:

thresholded_map, threshold = threshold_stats_img(
    stat_img=min_stat_map,
    alpha=0.001,            # FDR level (start here)
    height_control='fpr',  # use 'bonferroni' for more conservative FWER control
    cluster_threshold=5,   # use >0 to filter small clusters
    two_sided=True        # or True if you want two-sided test
)

output_path = os.path.join(output_dir, "min_stat_conjunction_fprcorrected_clusterThr5.nii.gz")
thresholded_map.to_filename(output_path)

In [ ]:

# Display the thresholded statistical map on a glass brain
display = plotting.plot_glass_brain(
    thresholded_map,
    threshold=None,          
    colorbar=True,
    display_mode='lyrz',     # Combine lateral, coronal, sagittal, axial views
    plot_abs=False,          # Set True if you want both positive & negative activations
    cmap='cold_hot',         # Common for statistical maps
    title='FPR-corrected (α=0.001, cluster>5)',
)

# Save the glass brain figure as an image
fig_path = os.path.join(output_dir, "glassbrain_min_stat_conjunction_fprClusterThr5.png")
display.savefig(fig_path, dpi=300)
display.close()

print(f"Glass brain saved at: {fig_path}")

In [ ]:
output_path = os.path.join(output_dir, "min_stat_conjunction_fprcorrected_clusterThr10.nii.gz")



In [ ]:

create_output(
        output_path,
        cluster_extent=10,
        direction="pos",
        outdir=str(output_dir)
    )

In [ ]:
output_path2 = os.path.join(output_dir, "min_stat_conjunction_fprcorrected_clusterThr5.nii.gz")


In [ ]:
create_output(
        output_path2,
        cluster_extent=5,
        direction="pos",
        outdir=str(output_dir)
    )

In [ ]:
output_path3 = os.path.join(output_dir, "min_stat_conjunction_fdrcorrected.nii.gz")

create_output(
        output_path3,
        cluster_extent=0,
        direction="pos",
        outdir=str(output_dir)
    )

In [ ]:
thresholded_map, threshold = threshold_stats_img(
    stat_img=min_stat_map,
    alpha=0.05,            # FDR level (start here)
    height_control='bonferroni',  # use 'bonferroni' for more conservative FWER control
    cluster_threshold=0,   # use >0 to filter small clusters
    two_sided=True        # or True if you want two-sided test
)

print(threshold)

output_path = os.path.join(output_dir, "min_stat_conjunction_bonfcorrected.nii.gz")
thresholded_map.to_filename(output_path)

# Display the thresholded statistical map on a glass brain
display = plotting.plot_glass_brain(
    thresholded_map,
    threshold=None,         
    colorbar=True,
    display_mode='lyrz',     # Combine lateral, coronal, sagittal, axial views
    plot_abs=False,          # Set True if you want both positive & negative activations
    cmap='cold_hot',         # Common for statistical maps
    title='bonferroni-corrected',
)

# Save the glass brain figure as an image
fig_path = os.path.join(output_dir, "glassbrain_min_stat_conjunction_bonfcorrected.png")
display.savefig(fig_path, dpi=300)
display.close()

print(f"Glass brain saved at: {fig_path}")


output_path4 = os.path.join(output_dir, "min_stat_conjunction_bonfcorrected.nii.gz")

create_output(
        output_path4,
        cluster_extent=0,
        direction="pos",
        outdir=str(output_dir)
    )